In [ ]:
!pip install -q langchain langchain-openai langchain-chroma tiktoken langchain-community


In [ ]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


In [ ]:
platform_rules = """
1. Pravo na postavljanje oglasa imaju isključivo lica koja su punoletna (18+ godina) i koja su prošla email verifikaciju.
2. Korisnici koji su prošli identifikaciju (KYC — upload ličnog dokumenta) dobijaju oznaku "Verifikovan" na profilu i imaju veće poverenje među drugim korisnicima.
3. Postavljanje oglasa je besplatno. Promotivne opcije se naplaćuju u kreditima: FEATURED 500 RSD (7 dana, vrh pretrage), PRIORITY 250 RSD (3 dana), HIGHLIGHTED 100 RSD (30 dana, vizuelni badge).
4. Dopuna kredita vrši se uplatom na zvanični račun platforme (Dimitrije Mitic, 265-0000006785327-58) uz obavezno korišćenje jedinstvenog poziva na broj koji se generiše na stranici `/credit`.
5. Krediti nisu povratni nakon što su iskorišćeni za aktivaciju promocije. Ako promocija nije konzumirana, korisnik može tražiti povraćaj u roku od 7 dana na mejl podrške.
6. Depozit koji zakupac eventualno uplaćuje zakupodavcu obavezno se vraća po isteku ugovora, ukoliko predmet iznajmljivanja nije oštećen i ukoliko je vraćen u dogovorenom roku.
7. Stanje iznajmljene stvari (fotografije, funkcionalnost) obavezno se dokumentuje pri primopredaji — i preporučuje se razmena tih fotografija u chat-u platforme kao dokaz.
8. Ukoliko stanodavac (vlasnik stvari) otkaže već prihvaćen ugovor bez opravdanog razloga, zakupac ima pravo na prioritetnu podršku i eventualnu naknadu troškova dokazanih kroz chat istoriju.
9. Otkazni rok za već prihvaćen ugovor je minimum 48 sati pre datuma početka — kasnije otkazivanje može rezultirati negativnom recenzijom i suspenzijom naloga kod ponavljanja.
10. Oštećenje iznajmljene stvari mora biti prijavljeno u roku od 48 sati od primopredaje. Prijava se vrši kroz chat platforme i mejlom na izdajemiznajmljujem.rs@gmail.com sa fotografijama.
11. Platforma ne posreduje u plaćanju naknade za rental (iznajmljivanje) — to se dogovara direktno između korisnika. Platforma naplaćuje isključivo promotivne opcije putem sistema kredita.
12. Zabranjeno je oglašavanje: oružja, narkotika, životinja, ljudskih organa, krađene robe, falsifikata, pirotehnike, alkohola licima mlađim od 18 godina, kao i bilo kakvih nelegalnih usluga.
13. Korisnik ima pravo na brisanje naloga i svih ličnih podataka u skladu sa Zakonom o zaštiti podataka o ličnosti (ZZPL) i GDPR-om — zahtev se šalje na izdajemiznajmljujem.rs@gmail.com.
14. Kasno vraćanje iznajmljene stvari može rezultirati dodatnom naknadom za svaki započet dan kašnjenja prema ceni iz originalnog oglasa.
15. Sporovi oko ugovora rešavaju se najpre direktno kroz chat platforme. Ukoliko korisnici ne postignu dogovor, slučaj se eskaliraju na izdajemiznajmljujem.rs@gmail.com — admini posreduju u roku od 3 radna dana.
16. Prijava sumnje na prevaru (traženje uplate na privatne račune van /credit stranice, lažni oglasi, pretnje) mora biti bezuslovna — suspenzija naloga je trenutna, a slučaj se predaje nadležnim organima ako postoje elementi krivičnog dela.
17. Recenzije se mogu ostaviti isključivo nakon završenog ugovora, u roku od 30 dana. Lažne recenzije, međusobno "nameštanje" i vređanja se brišu bez upozorenja.
18. Prava potrošača u skladu sa Zakonom o zaštiti potrošača — pravo na reklamaciju neispravne usluge, pravo na informisanost, pravo na zaštitu od nepoštene poslovne prakse — primenjuju se na sve komercijalne transakcije preko platforme.
"""

In [ ]:
with open("pravila_izdavanja.txt", "w", encoding = "utf-8") as f:
  f.write(platform_rules)

In [ ]:
loader = TextLoader("pravila_izdavanja.txt")
document = loader.load()
print("Successfuly loaded file")

Successfuly loaded file


In [ ]:
from dotenv import load_dotenv
load_dotenv()

CHUNKING

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 150,
    chunk_overlap = 20
)
splitted_docs = text_splitter.split_documents(document)
print(f"Splitted on {len(splitted_docs)} chunks")

Splitted on 34 chunks


SBERT embedding




In [ ]:
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

vectors = Chroma.from_documents(
    documents = splitted_docs,
    embedding=embeddings_model,
    persist_directory = "./chroma_baza_stanova"
)

In [ ]:
retriever = vectors.as_retriever(search_kwargs={"k": 2})

In [ ]:
llm = ChatOpenAI(model = "gpt-4o-mini", temperature=0.6)

prompt_template = PromptTemplate(
    template = """Ti si gotivni ljubazni dobri agent za korisničku podršku sajta 'Izdajem Iznajmljujem'.
Na osnovu ponuđenog KONTEKSTA, odgovori na PITANJE korisnika.
Ako u KONTEKSTU nema odgovora, reci: "Nažalost, nemam tu informaciju u pravilniku." Nemoj nikada da izmišljaš pravila.

KONTEKST:
{context}

PITANJE KORISNIKA: {question}

ODGOVOR:""",
    input_variables=["context", "question"]
)

In [ ]:
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()

)

In [ ]:
from typing import TypedDict, List, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
import operator


In [ ]:
# question = "Da li se placa  za oglas na vašem sajtu?"
# print(f"Korisnik pita: {question}")

# answer = rag_chain.invoke(question)
# print(f"Bot odgovara:\n{answer}")

Graf u LangGraph funkcionise tako sto svaki node (cvor) prima trenutno stanje koje obradjuje i azurira zatim prosledjuje sledecem cvoru. AgentState definise koje se informacije prenose u state-u.
AgentState pruza jednostavan nacin da se sacuva i upravlja state-om AI agentom u realnom vremenu. Informacije koje prenosimo su: question, dokumenats, is_relevatn i asnwer.

In [ ]:
class AgentState(TypedDict):
    question: str
    documents: List[str]
    expanded_queries: List[str]
    chat_history: Annotated[List[str], operator.add]
    is_relevant: str
    answer: str

In [ ]:
def query_expansion_node(state):
    question = state["question"]

    prompt = f"""Preformuliši sledeće korisničko pitanje u 3 različita, detaljnija pitanja na srpskom jeziku, kako bi olakšao pretragu pravila i dokumenata u bazi.
    Odgovori ISKLJUČIVO spiskom, svaki upit u novom redu, bez brojeva i bez dodatnog teksta.
    Originalno pitanje: {question}"""

    # LLM generiše 3 nova pitanja
    response = llm.invoke(prompt).content.strip()

    # Sečemo odgovor u listu stringova
    queries = response.split('\n')

    # Obavezno dodajemo i originalno pitanje u listu, zlu ne trebalo
    queries.append(question)

    return {"expanded_queries": queries}

`retrieve_node` je izvrsni cvor koji je zaduzen za pronalazenje relevantnih informacija. Zadatak mu je da pomocu `retriever` objekta komunicira sa vektroskom bazom (Chroma).

Pitanje se konvertuje u vektor
(embedding) i pronalazi 2 (k=2) najslicnija pasusa iz Chroma koristeci kosinusnu slicnost.

Iteracija kroz dokument i izvlacenje cistog teksta (page_content) u novu listu stringova

In [ ]:
def retrieve_node(state):
  question = state["question"]
  history = "\n".join(state.get("chat_history", [])[-5:])

  expansion_prompt = f"""Korisnik je postavio kratko pitanje: '{question}'.
    Prethodni razgovor: {history}
    Tvoj zadatak je da preformulišeš ovo pitanje na 3 različita načina (sinonimi, profesionalniji rečnik), kako bismo što lakše pronašli odgovor u bazi pravila o iznajmljivanju.
    Vrati ISKLJUČIVO ta 3 pitanja, svako u novom redu, bez ikakvog dodatnog teksta, rednih brojeva ili znakova."""


  expanded_queries_text = llm.invoke(expansion_prompt).content
  expanded_queries = expanded_queries_text.strip().split('\n')

  expanded_queries.append(question)
  print(f"-> Prošireni upiti za pretragu:\n{expanded_queries_text}")

  all_docs = []
  seen_content = set()

  for q in expanded_queries:
    if not q.strip(): continue
    docs = retriever.invoke(q)
    for d in docs:
      if d.page_content not in seen_content:
        seen_content.add(d.page_content)
        all_docs.append(d.page_content)

  return {"documents": all_docs[:4]}

`grade_node` je evaluacioni cvor koji implementira napredni koncept Self-RAG. U ovom cvoru LLM se koristi iskljucivo za proveru cinjenica

Njegov zadatak je da uporedi *question* iz state-a sa dokumentima dobijenim iz `retrieve_node` i proceni da li dokumenti sadrze bar delimican odgovor na dobijeno pitanje. Cvor prihvata trenutno stanje - *state*

1.   Izvlaci pitanje iz *state*
2.   Spaja listu dobijenih dokumenta u jedan string - *context*
3.   Kreira se prompt koji ima binaran izlaz
4.   Poziva se LLM da izvrsi prompt, ako tekst sadrzi odgovor na pitanje, postavlja da je pitanje relevantno i obrnuto.






In [ ]:
def grade_node(state):
  question = state["question"]
  context = "\n".join(state["documents"])

  prompt = f"""Proveri da li TEKST sadrži odgovor na PITANJE.
    Odgovori ISKLJUČIVO sa: yes ili no.
    TEKST: {context}
    PITANJE: {question}"""

  decision = llm.invoke(prompt).content.strip().lower()

  if "yes" in decision:
    return {"is_relevant": "yes"}
  else:
    return {"is_relevant": "no"}

`generate_node` je izvršni cvor odgovoran za sintezu konacnog teksta, baziranu isključivo na kontekstu obezbeđenom putem RAG (Retrieval-Augmented Generation) metode.

In [ ]:
def generate_node(state):
  context = "\n\n".join(state["documents"])
  question = state["question"]
  history = "\n".join(state.get("chat_history", []))

  prompt = f"""Ti si gotivan ljubazan asistent na sajtu Izdajem Iznajmljujem.
  Ovo je prethodna istorija razgovora sa ovim korisnikom:
  {history}
  Na osnovu KONTEKSTA, odgovori na PITANJE.
  KONTEKST: {context}
  PITANJE: {question}"""

  answer = llm.invoke(prompt).content

  new_interaction = [f"Korisnik: {question}", f"Bot: {answer}"]
  return {"answer": answer, "chat_history": new_interaction}

`chat_node` sluzi kao direktan kanal (bypass) ka osnovnom jezickom modelu, zaduzen je za obradu opstih upita koji nisu vezani za domen.

Primer:

*   question: "Kako si?"
*   answer: "Super sam, kako si ti?"

Kada router detektuje da pitanje nije tehnicko ili nije vezano za koriscenje platforme, sistem preskace Chroma - vektorsku bazu i dozvoljava modelu da slobodno komunicira sa korisnikom, koristeci svoje predtrenirano znanje.



In [ ]:
def chat_node(state):
  question = state["question"]
  history = "\n".join(state.get("chat_history",[]))

  prompt = f"""Ti si gotivan ljubazan i dobar asistent na sajtu Izdajem Iznajmljujem.
  Ovo je prethodna istorija razgovora sa ovim korisnikom:
  {history}
  Korisnik ti kaže: '{question}'. Odgovori kratko i prijateljski."""

  answer = llm.invoke(prompt).content
  new_interaction = [f"Korisnik: {question}", f"Bot: {answer}"]
  return {"answer": answer, "chat_history": new_interaction}

`escalate_node` je Fallback mehanizam. U pitanju je deterministicki cvor koji vraca unapred definisani tekstualni odgovor (ne poziva LLM).

In [ ]:
def escalate_node(state):
  answer = "Nažalost, nemam tu informaciju u pravilniku. Da li želiš da te povežem sa operaterom?"
  new_interaction = [f"Korisnik: {state['question']}", f"Bot: {answer}"]
  return {"answer": answer, "chat_history": new_interaction}


Router je funkcija za uslovno usmeravanje unutar grafa. Zadatak je da klasifikuje namere korisnika i usmeravanje na sledeci cvor. Sprecava nepotrebno pretrazivanje vektorske baze (RAG) ukoliko se pitanje ne tice domena platforme. Time ustedi tokene i vreme.  
*   Cita question (pitanje iz state-a)
*   Kreira prompt za LLM koji binarno odlucuje da li se odnosi na domen platforme ili ne
*   Ako vrati "DA", preusmerava graf na retrieve_node, odnosno na pretragu dokumentacije
*   Ako vrati "NE", preusmerava graf na chat_node koji sluzi za obradu opstih upita (small talk / out of domain queries)





In [ ]:
def router(state):
  question = state["question"]
  history = "\n".join(state.get("chat_history", [])[-5:]) # Gledamo samo zadnjih 5 poruka da uštedimo tokene

  prompt = f"""Da li se ovo pitanje odnosi na pravila, oglase, depozite, plaćanje, kredite ili funkcionisanje sajta za nekretnine?
  Uzmi u obzir i KONTEKST prethodnog razgovora (možda se korisnik nadovezuje na prethodnu temu).
  Prethodni razgovor: {history}
  Pitanje: {question}
  Odgovori ISKLJUČIVO sa DA ili NE."""

  decision = llm.invoke(prompt).content.strip().lower()

  if "da" in decision:
    return "retrieve_node"
  else:
    return "chat_node"

`check_relevance` je staticka funkcija (ne koristi LLM) za uslovno usmeravanje (conditional edge). Ona donosi konacnu odluku o relevantnosti na osnovu ocene koju je dodelio `grade_node`.

In [ ]:
def check_relevance(state):
    if state["is_relevant"] == "yes":
        return "generate_node"
    else:
        return "escalate_node"

In [ ]:
workflow = StateGraph(AgentState)

workflow.add_node("retrieve_node", retrieve_node)
workflow.add_node("grade_node", grade_node)
workflow.add_node("generate_node", generate_node)
workflow.add_node("escalate_node", escalate_node)
workflow.add_node("chat_node", chat_node)

In [ ]:
workflow.add_conditional_edges(
    START,
    router,
    {
        "retrieve_node": "retrieve_node",
        "chat_node": "chat_node"
    }
)

In [ ]:
workflow.add_edge("retrieve_node", "grade_node")

In [ ]:
workflow.add_conditional_edges(
    "grade_node",
    check_relevance,
    {
        "generate_node": "generate_node",
        "escalate_node": "escalate_node"
    }
)

In [ ]:
workflow.add_edge("generate_node", END)
workflow.add_edge("escalate_node", END)
workflow.add_edge("chat_node", END)

In [ ]:
memory = MemorySaver()

In [ ]:
agent_app = workflow.compile(checkpointer= memory)
print("Graf je uspesno kompajliran na svim cvorovima")

Graf je uspesno kompajliran na svim cvorovima


# OVDE IDE USER ID SA BACK-A

In [ ]:
# config = {"configurable": {"thread_id": "USER_ID"}}

# print("\n--- PRVO PITANJE ---")
# q1 = "Koliki je rok za povraćaj kredita ako promocija nije iskorišćena?"
# print(f"KORISNIK: {q1}")
# res1 = agent_app.invoke({"question": q1}, config=config)
# print(f"BOT: {res1['answer']}")

# print("\n--- DRUGO PITANJE (Nadovezivanje - zahteva memoriju!) ---")
# # Obrati pažnju: Ne spominjemo "kredit" niti "povraćaj", samo kažemo "A na koji mejl to šaljem?"
# q2 = "A na koji mejl to šaljem?"
# print(f"KORISNIK: {q2}")
# # Koristimo ISTI config da bi znao da je to isti korisnik
# res2 = agent_app.invoke({"question": q2}, config=config)
# print(f"BOT: {res2['answer']}")

In [ ]:
config = {"configurable": {"thread_id": "test-user-1"}}

question_1 = "Da li se placa za oglas na vašem sajtu?"
print(f"KORISNIK: {question_1}")
result_1 = agent_app.invoke({"question": question_1}, config=config)
print(f"BOT: {result_1['answer']}\n")
print("-" * 50)

# Test 2: Pitanje za običan razgovor (Testiramo skretničara!)
question_2 = "Da li znas kvadratnu jednacinu?"
print(f"\nKORISNIK: {question_2}")
result_2 = agent_app.invoke({"question": question_2}, config=config)
print(f"BOT: {result_2['answer']}")

In [ ]:
config = {"configurable": {"thread_id": "test-user-2"}}

print("\n--- TEST 1: Pitanje koje postoji u pravilniku (RAG -> GRADE -> GENERATE) ---")
q1 = "Koliko dana traje PRIORITY oglas i koliko košta?"
print(f"KORISNIK: {q1}")
print(f"BOT: {agent_app.invoke({'question': q1}, config=config)['answer']}")

print("\n--- TEST 2: Pitanje o sajtu, ali NE POSTOJI u pravilniku (RAG -> GRADE -> ESCALATE) ---")
q2 = "Kako da promenim temu sajta u crnu?"
print(f"KORISNIK: {q2}")
print(f"BOT: {agent_app.invoke({'question': q2}, config=config)['answer']}")

print("\n--- TEST 3: Običan razgovor (START -> CHAT) ---")
q3 = "Da li znas kvadratnu jednacinu?"
print(f"KORISNIK: {q3}")
print(f"BOT: {agent_app.invoke({'question': q3}, config=config)['answer']}")

In [ ]:
!pip install pyngrok
from pyngrok import ngrok

In [ ]:
public_url = ngrok.connect(8000)

ERROR:pyngrok.process.ngrok:t=2026-04-20T21:00:23+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-20T21:00:23+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.